In [7]:
!pip install -U pip setuptools==68.0.0 wheel

!pip install \
    numpy==1.24.3 \
    pandas==2.0.3 \
    pytorch_lightning==2.0.6 \
    scikit_learn==1.3.1 \
    torch==2.0.1 \
    tqdm==4.65.0 \
    transformers==4.33.3 \
    peft==0.5.0 \
    accelerate==0.23.0

!pip install microformer-mgm
!pip install optuna
!pip uninstall -y jax jaxlib

In [9]:
import os
import torch
import optuna
import pandas as pd
import numpy as np

from pickle import load, dump

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

from transformers import (
    GPT2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from transformers.trainer_callback import EarlyStoppingCallback

from mgm.src.MicroCorpus import SequenceClassificationDataset

ValueError: Unable to avoid copy while creating an array as requested.
If using `np.array(obj, copy=False)` replace it with `np.asarray(obj)` to allow a copy when needed (no behavior change in NumPy 1.x).
For more details, see https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword.

In [ ]:
import zipfile

with zipfile.ZipFile('MGM_NHANES_DAPT.zip', 'r') as zip_ref:
    zip_ref.extractall('MGM_NHANES_DAPT')

In [ ]:
!mgm construct \
-i asv_genus.csv \
-o corpus.pkl

In [ ]:
from pickle import load

corpus = load(open("corpus.pkl","rb"))

corpus[0]

In [ ]:
df_metadata = pd.read_csv(
    "df_metadata.csv"
)


df_metadata.head()

In [ ]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


folds = list(
    cv.split(
        df_metadata,
        df_metadata["disease_status"],
        groups=df_metadata["pair_id"]
    )
)


print(len(folds))

In [ ]:
def set_trainable_layers(
    model,
    mode
):

    for p in model.parameters():
        p.requires_grad=False


    if mode=="all":

        for p in model.parameters():
            p.requires_grad=True


    elif mode=="last2":

        for layer in model.transformer.h[-2:]:
            for p in layer.parameters():
                p.requires_grad=True


    elif mode=="last4":

        for layer in model.transformer.h[-4:]:
            for p in layer.parameters():
                p.requires_grad=True


    for p in model.score.parameters():
        p.requires_grad=True

In [ ]:
def train_fold(
    train_idx,
    val_idx,
    params,
    fold
):


    train_meta = df_metadata.iloc[train_idx]

    val_meta = df_metadata.iloc[val_idx]


    train_ids = train_meta["Run"].values
    val_ids = val_meta["Run"].values



    train_labels = (
        train_meta
        .set_index("Run")["disease_status"]
    )


    val_labels = (
        val_meta
        .set_index("Run")["disease_status"]
    )


    train_positions = [
        corpus.data.index.get_loc(x)
        for x in train_ids
    ]


    val_positions = [
        corpus.data.index.get_loc(x)
        for x in val_ids
    ]



    train_dataset = SequenceClassificationDataset(
        corpus[train_positions]["input_ids"],
        corpus[train_positions]["attention_mask"],
        torch.tensor(train_labels.values)
    )


    val_dataset = SequenceClassificationDataset(
        corpus[val_positions]["input_ids"],
        corpus[val_positions]["attention_mask"],
        torch.tensor(val_labels.values)
    )


    # cargar DAPT
    model = GPT2ForSequenceClassification.from_pretrained(
        "MGM_NHANES_DAPT",
        num_labels=2
    )


    set_trainable_layers(
        model,
        params["layers"]
    )


    args = TrainingArguments(

        output_dir=f"optuna_fold_{fold}",

        learning_rate=params["learning_rate"],

        weight_decay=params["weight_decay"],

        warmup_ratio=params["warmup_ratio"],

        num_train_epochs=20,

        per_device_train_batch_size=8,

        evaluation_strategy="epoch",

        save_strategy="no",

        logging_steps=10,

        report_to="none"
    )


    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset
    )


    trainer.train()


    preds = trainer.predict(
        val_dataset
    )


    probs = torch.softmax(
        torch.tensor(preds.predictions),
        dim=1
    )[:,1]


    auc = roc_auc_score(
        val_labels.values,
        probs.detach().numpy()
    )

    del model
    del trainer

    torch.cuda.empty_cache()

    return auc

In [ ]:
def evaluate_params(params):

    aucs = []

    for fold, (train_idx, val_idx) in enumerate(folds):

        print(f"\nFold {fold+1}")

        auc = train_fold(
            train_idx=train_idx,
            val_idx=val_idx,
            params=params,
            fold=fold
        )

        print(f"AUC = {auc:.4f}")

        aucs.append(auc)

    print("\n====================")
    print("Mean AUC:", np.mean(aucs))
    print("Std AUC :", np.std(aucs))

    return aucs

In [ ]:
def objective(trial):

    params = {

        "learning_rate":
        trial.suggest_float(
            "learning_rate",
            1e-5,
            5e-4,
            log=True
        ),

        "weight_decay":
        trial.suggest_categorical(
            "weight_decay",
            [0,0.01,0.1]
        ),

        "warmup_ratio":
        trial.suggest_categorical(
            "warmup_ratio",
            [0,0.1,0.2]
        ),

        "layers":
        trial.suggest_categorical(
            "layers",
            ["all","last4","last2"]
        )
    }


    try:

        scores=[]

        for fold,(train_idx,val_idx) in enumerate(folds):

            auc=train_fold(
                train_idx,
                val_idx,
                params,
                fold
            )

            scores.append(auc)


        return np.mean(scores)


    except torch.cuda.OutOfMemoryError:

        torch.cuda.empty_cache()

        return 0.0

In [ ]:
study = optuna.create_study(
    direction="maximize"
)


study.optimize(
    objective,
    n_trials=30
)

In [ ]:
study.best_params

In [ ]:
pd.DataFrame(
    study.trials_dataframe()
).to_csv(
    "optuna_results.csv",
    index=False
)

In [ ]:
params_last4 = {
    "learning_rate": 3.3410221978461924e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.2,
    "layers": "last4"
}

auc_last4 = evaluate_params(params_last4)

In [ ]:
params_all = {
    "learning_rate": 4.59324400326387e-4,
    "weight_decay": 0.1,
    "warmup_ratio": 0.1,
    "layers": "all"
}

auc_all = evaluate_params(params_all)

In [ ]:
print("\nRESULTADOS")

print(f"last4: {np.mean(auc_last4):.4f} ± {np.std(auc_last4):.4f}")
print(f"all  : {np.mean(auc_all):.4f} ± {np.std(auc_all):.4f}")

In [ ]:
# eliminar muestras de metadata que no existen en el corpus
df_final = df_metadata[
    df_metadata["Run"].isin(corpus.data.index)
].copy()


print("Muestras finales:", len(df_final))
print(df_final["disease_status"].value_counts())

In [ ]:
final_positions = [
    corpus.data.index.get_loc(x)
    for x in df_final["Run"]
]

In [ ]:
dataset_final = SequenceClassificationDataset(

    corpus[final_positions]["input_ids"],

    corpus[final_positions]["attention_mask"],

    torch.tensor(
        df_final["disease_status"].values,
        dtype=torch.long
    )
)

In [ ]:
print(len(dataset_final))
print(dataset_final[0])

In [ ]:
model_final = GPT2ForSequenceClassification.from_pretrained(
    "MGM_NHANES_DAPT",
    num_labels=2
)

In [ ]:
best_params = {
    "learning_rate": 4.59324400326387e-4,
    "weight_decay": 0.1,
    "warmup_ratio": 0.1
}

In [ ]:
training_args_final = TrainingArguments(

    output_dir="MGM_NHANES_finetuned_all",

    learning_rate=best_params["learning_rate"],

    weight_decay=best_params["weight_decay"],

    warmup_ratio=best_params["warmup_ratio"],

    num_train_epochs=50,

    per_device_train_batch_size=8,

    save_strategy="epoch",

    logging_strategy="epoch",

    report_to="none"
)

In [ ]:
trainer_final = Trainer(

    model=model_final,

    args=training_args_final,

    train_dataset=dataset_final
)


trainer_final.train()

In [ ]:
trainer_final.save_model(
    "MGM_NHANES_finetuned_all"
)